<a href="https://colab.research.google.com/github/armandochernandez-ai/Curso-python-slava/blob/main/Desaparecidos/Extractor_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Método DIRECTO - Analizar el HTML del mapa y buscar datos
import requests
import json
import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os
from google.colab import drive
import numpy as np

# 1. Montar Google Drive
print("Montando Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 2. Crear carpeta
drive_folder = '/content/drive/My Drive/Desaparecidos_Jalisco'
os.makedirs(drive_folder, exist_ok=True)
print(f"Carpeta: {drive_folder}")

# 3. URL del mapa
url = "https://mapajaliscodesapariciones.rys2000dev.workers.dev/"

# 4. Obtener el HTML
print("\nDescargando HTML del mapa...")
try:
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    html_content = response.text
    print(f"✅ HTML descargado ({len(html_content)} caracteres)")
except Exception as e:
    print(f"❌ Error descargando HTML: {e}")
    html_content = ""

# 5. Guardar HTML para análisis
if html_content:
    with open(f'{drive_folder}/mapa_html.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    print(f"📄 HTML guardado: {drive_folder}/mapa_html.html")

# 6. Buscar datos GeoJSON/JSON en el HTML
print("\n🔍 Buscando datos GeoJSON/JSON en el código...")

# Patrones para encontrar URLs de datos
patterns = [
    r'"url"\s*:\s*["\']([^"\']+\.(?:geojson|json))["\']',
    r'src=["\']([^"\']+\.(?:geojson|json))["\']',
    r'data-url=["\']([^"\']+\.(?:geojson|json))["\']',
    r'load\(\s*["\']([^"\']+\.(?:geojson|json))["\']',
    r'fetch\(\s*["\']([^"\']+\.(?:geojson|json))["\']',
    r'L\.geoJSON\(\s*["\']([^"\']+\.(?:geojson|json))["\']',
    r'addLayer\(\s*["\']([^"\']+\.(?:geojson|json))["\']',
    r'https?://[^"\']+\.(?:geojson|json)'
]

all_urls = []
for pattern in patterns:
    try:
        urls = re.findall(pattern, html_content, re.IGNORECASE)
        all_urls.extend(urls)
    except:
        continue

# Eliminar duplicados y limpiar
unique_urls = list(set(all_urls))
unique_urls = [u for u in unique_urls if u and len(u) > 10]

print(f"Encontradas {len(unique_urls)} URLs potenciales de datos")

# 7. Analizar archivos JavaScript
print("\n🔍 Analizando archivos JavaScript...")
js_pattern = r'<script[^>]+src=["\']([^"\']+\.js)["\']'
js_urls = re.findall(js_pattern, html_content, re.IGNORECASE)

for i, js_url in enumerate(js_urls[:10]):  # Analizar solo primeros 10 JS
    try:
        if not js_url.startswith('http'):
            if js_url.startswith('//'):
                js_url = 'https:' + js_url
            elif js_url.startswith('/'):
                js_url = 'https://mapajaliscodesapariciones.rys2000dev.workers.dev' + js_url
            else:
                js_url = url + '/' + js_url

        print(f"  {i+1}. Analizando: {js_url}")
        js_response = requests.get(js_url, headers=headers, timeout=10)
        if js_response.status_code == 200:
            js_content = js_response.text

            # Buscar URLs de datos en JS
            for pattern in patterns:
                js_urls_found = re.findall(pattern, js_content, re.IGNORECASE)
                for found_url in js_urls_found:
                    if found_url and found_url not in unique_urls and len(found_url) > 10:
                        unique_urls.append(found_url)

    except Exception as e:
        print(f"    ✗ Error: {str(e)[:50]}")

# 8. Descargar y procesar datos encontrados
print(f"\n📥 Descargando archivos de datos...")

all_data = []
downloaded_files = []
failed_urls = []

for i, data_url in enumerate(unique_urls[:50]):  # Limitar a 50 URLs
    try:
        # Completar URL si es relativa
        original_url = data_url
        if not data_url.startswith('http'):
            if data_url.startswith('//'):
                data_url = 'https:' + data_url
            elif data_url.startswith('/'):
                data_url = 'https://mapajaliscodesapariciones.rys2000dev.workers.dev' + data_url
            else:
                data_url = url + '/' + data_url

        print(f"  {i+1}. Descargando: {data_url[:80]}...")

        response = requests.get(data_url, headers=headers, timeout=15)

        if response.status_code == 200:
            # Intentar parsear como JSON/GeoJSON
            try:
                data = response.json()

                # Guardar archivo
                filename = f"data_{i+1}_{original_url.split('/')[-1][:50]}"
                if not filename.endswith('.json'):
                    filename += '.json'

                filepath = f'{drive_folder}/{filename}'

                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

                downloaded_files.append({
                    'url': data_url,
                    'filename': filename,
                    'size_kb': len(response.content) / 1024,
                    'type': 'json'
                })

                # Procesar datos según tipo
                if isinstance(data, dict):
                    if data.get('type') == 'FeatureCollection':
                        features = data.get('features', [])
                        print(f"    ✅ GeoJSON con {len(features)} features")

                        for feature in features:
                            if isinstance(feature, dict):
                                all_data.append(feature)

                    elif 'features' in data:
                        features = data.get('features', [])
                        print(f"    ✅ Formato con {len(features)} features")

                        for feature in features:
                            if isinstance(feature, dict):
                                all_data.append(feature)

                    else:
                        # Podría ser un diccionario con datos
                        all_data.append(data)

                elif isinstance(data, list):
                    print(f"    ✅ Lista con {len(data)} elementos")
                    all_data.extend(data)

            except json.JSONDecodeError:
                print(f"    ⚠️ No es JSON válido, guardando como texto")
                filename = f"data_{i+1}_{original_url.split('/')[-1][:50]}"
                filepath = f'{drive_folder}/{filename}'
                with open(filepath, 'wb') as f:
                    f.write(response.content)

                downloaded_files.append({
                    'url': data_url,
                    'filename': filename,
                    'size_kb': len(response.content) / 1024,
                    'type': 'other'
                })

        else:
            print(f"    ❌ Error HTTP {response.status_code}")
            failed_urls.append(data_url)

    except Exception as e:
        print(f"    ❌ Error: {str(e)[:100]}")
        failed_urls.append(data_url)

# 9. Procesar y guardar datos extraídos
print(f"\n💾 Procesando {len(all_data)} registros extraídos...")

if all_data:
    # Convertir a DataFrame/GeoDataFrame
    processed_data = []

    for i, item in enumerate(all_data):
        try:
            record = {'id': i, 'tipo_dato': type(item).__name__}

            if isinstance(item, dict):
                # Extraer propiedades si es una feature de GeoJSON
                if 'properties' in item:
                    record.update(item['properties'])
                    record['tipo'] = 'feature'

                # Extraer geometría si existe
                if 'geometry' in item and item['geometry']:
                    geom = item['geometry']
                    record['tipo_geometria'] = geom.get('type')

                    if geom.get('type') == 'Point' and geom.get('coordinates'):
                        coords = geom['coordinates']
                        if len(coords) >= 2:
                            record['longitud'] = float(coords[0])
                            record['latitud'] = float(coords[1])
                            record['coordenadas'] = f"{coords[1]}, {coords[0]}"

                    elif geom.get('type') == 'Polygon' and geom.get('coordinates'):
                        # Para polígonos, usar el centroide aproximado
                        try:
                            all_coords = []
                            for ring in geom['coordinates']:
                                for coord in ring:
                                    if len(coord) >= 2:
                                        all_coords.append(coord)

                            if all_coords:
                                coords_array = np.array(all_coords)
                                center = coords_array.mean(axis=0)
                                record['longitud_centro'] = float(center[0])
                                record['latitud_centro'] = float(center[1])
                        except:
                            pass

                # Si no es una feature, copiar todos los campos
                else:
                    for key, value in item.items():
                        if isinstance(value, (str, int, float, bool, type(None))):
                            record[key] = value
                        else:
                            record[key] = str(value)

            elif isinstance(item, (str, int, float)):
                record['valor'] = item

            processed_data.append(record)

        except Exception as e:
            print(f"  ⚠️ Error procesando registro {i}: {str(e)[:50]}")

    # Crear DataFrame
    df = pd.DataFrame(processed_data)

    # Guardar en múltiples formatos
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

    # CSV
    csv_path = f'{drive_folder}/datos_extraidos_{timestamp}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✅ CSV guardado: {csv_path} ({len(df)} registros, {len(df.columns)} columnas)")

    # JSON
    json_path = f'{drive_folder}/datos_extraidos_{timestamp}.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON guardado: {json_path}")

    # GeoJSON si hay coordenadas
    geo_created = False
    if 'latitud' in df.columns and 'longitud' in df.columns:
        # Filtrar filas con coordenadas válidas
        valid_coords = df.dropna(subset=['latitud', 'longitud'])

        if not valid_coords.empty and len(valid_coords) > 0:
            try:
                # Crear geometrías
                geometry = [Point(xy) for xy in zip(valid_coords['longitud'], valid_coords['latitud'])]

                # Crear GeoDataFrame
                gdf = gpd.GeoDataFrame(valid_coords, geometry=geometry, crs='EPSG:4326')

                # Guardar como GeoJSON
                geojson_path = f'{drive_folder}/datos_geo_{timestamp}.geojson'
                gdf.to_file(geojson_path, driver='GeoJSON')
                print(f"✅ GeoJSON guardado: {geojson_path} ({len(gdf)} puntos)")
                geo_created = True
            except Exception as e:
                print(f"⚠️ Error creando GeoJSON: {e}")

    # Resumen
    print(f"\n📊 RESUMEN DE DATOS:")
    print(f"  Total registros procesados: {len(df)}")

    if 'latitud' in df.columns:
        with_coords = df['latitud'].notna().sum()
        print(f"  Registros con coordenadas: {with_coords}")

    if 'tipo_geometria' in df.columns:
        geom_counts = df['tipo_geometria'].value_counts()
        print(f"  Tipos de geometría:")
        for geom_type, count in geom_counts.items():
            print(f"    - {geom_type}: {count}")

    print(f"\n  Primeras columnas: {list(df.columns)[:15]}")

else:
    print("⚠️ No se encontraron datos procesables en las URLs")

# 10. Guardar reporte de archivos descargados
if downloaded_files:
    report_df = pd.DataFrame(downloaded_files)
    report_path = f'{drive_folder}/reporte_descargas_{timestamp}.csv'
    report_df.to_csv(report_path, index=False)
    print(f"\n📋 Reporte de descargas: {report_path}")
    print(f"   Total archivos descargados: {len(downloaded_files)}")

# 11. Búsqueda directa en el contenido HTML
print("\n🔍 Búsqueda directa en HTML para datos estructurados...")

# Buscar patrones específicos de datos en el HTML
data_patterns = {
    'coordenadas': r'[-+]?\d{1,3}\.\d+\s*,\s*[-+]?\d{1,3}\.\d+',
    'fechas_iso': r'\d{4}-\d{2}-\d{2}',
    'fechas_mex': r'\d{2}/\d{2}/\d{4}',
    'municipios': r'(?i)(Guadalajara|Zapopan|Tlaquepaque|Tonalá|Tlajomulco|El Salto|Ixtlahuacán|Chapala|Puerto Vallarta|Lagos de Moreno)',
    'numeros': r'\b\d{2,5}\b',
    'palabras_clave': r'(?i)(desaparecido|localizado|homicidio|fosa|feminicidio|municipio|colonia|edad|sexo)',
}

html_data_found = {}
for key, pattern in data_patterns.items():
    matches = re.findall(pattern, html_content)
    if matches:
        unique_matches = list(set(matches))
        html_data_found[key] = unique_matches[:100]  # Limitar a 100
        print(f"  {key}: {len(html_data_found[key])} encontrados")

# Guardar datos HTML
if html_data_found:
    html_data_path = f'{drive_folder}/datos_html_{timestamp}.json'
    with open(html_data_path, 'w', encoding='utf-8') as f:
        json.dump(html_data_found, f, ensure_ascii=False, indent=2)
    print(f"✅ Datos HTML guardados: {html_data_path}")

# 12. Búsqueda de scripts con datos embebidos
print("\n🔍 Buscando datos embebidos en scripts...")

# Buscar variables JavaScript que puedan contener datos
script_patterns = [
    r'var\s+(\w+)\s*=\s*(\{.*?\});',
    r'const\s+(\w+)\s*=\s*(\{.*?\});',
    r'let\s+(\w+)\s*=\s*(\{.*?\});',
    r'data\s*:\s*(\{.*?\})',
    r'features\s*:\s*\[.*?\]',
]

for pattern in script_patterns:
    matches = re.findall(pattern, html_content, re.DOTALL)
    if matches:
        print(f"  Encontrados {len(matches)} conjuntos de datos potenciales en scripts")

# 13. Listar archivos creados
print("\n" + "="*60)
print("¡ANÁLISIS COMPLETADO!")
print(f"📁 Archivos guardados en: {drive_folder}")
print("="*60)

# Listar archivos creados
print("\n📄 ARCHIVOS CREADOS:")
import glob
files = glob.glob(f'{drive_folder}/*')
files.sort(key=os.path.getmtime, reverse=True)

for file in files[:15]:  # Mostrar últimos 15
    size = os.path.getsize(file) / 1024
    filename = os.path.basename(file)
    print(f"  • {filename:<40} ({size:6.1f} KB)")

if len(files) > 15:
    print(f"  ... y {len(files) - 15} archivos más")

print("\n⚠️ NOTA: Si no se encontraron datos, el mapa probablemente carga")
print("  los datos dinámicamente mediante JavaScript. Para extraer esos datos")
print("  necesitaríamos usar Selenium o encontrar los endpoints de la API.")
print("\n🔍 Siguientes pasos posibles:")
print("  1. Inspeccionar manualmente el mapa en Chrome/Firefox (F12 -> Network)")
print("  2. Buscar peticiones a archivos .geojson o .json")
print("  3. Usar esas URLs para descargar los datos directamente")

# 14. Sugerencia para inspección manual
print("\n" + "="*60)
print("INSTRUCCIONES PARA INSPECCIÓN MANUAL:")
print("="*60)
print("1. Abre el mapa en Chrome/Firefox:")
print("   https://mapajaliscodesapariciones.rys2000dev.workers.dev/")
print("\n2. Presiona F12 para abrir DevTools")
print("\n3. Ve a la pestaña 'Network' (Red)")
print("\n4. Recarga la página (F5)")
print("\n5. Filtra por 'Fetch/XHR' o busca '.geojson', '.json'")
print("\n6. Copia las URLs de los archivos de datos")
print("\n7. Comparte esas URLs para descargarlas directamente")
print("="*60)

Montando Google Drive...
Mounted at /content/drive
Carpeta: /content/drive/My Drive/Desaparecidos_Jalisco

Descargando HTML del mapa...
✅ HTML descargado (35607 caracteres)
📄 HTML guardado: /content/drive/My Drive/Desaparecidos_Jalisco/mapa_html.html

🔍 Buscando datos GeoJSON/JSON en el código...
Encontradas 4 URLs potenciales de datos

🔍 Analizando archivos JavaScript...
  1. Analizando: https://cdn.jsdelivr.net/npm/chart.js
  2. Analizando: https://unpkg.com/maplibre-gl@3.6.1/dist/maplibre-gl.js

📥 Descargando archivos de datos...
  1. Descargando: https://mapajaliscodesapariciones.rys2000dev.workers.dev//fosas.geojson...
    ✅ GeoJSON con 128 features
  2. Descargando: https://mapajaliscodesapariciones.rys2000dev.workers.dev//colonias.geojson...
    ✅ GeoJSON con 6307 features
  3. Descargando: https://basemaps.cartocdn.com/gl/voyager-gl-style/style.json...
  4. Descargando: https://mapajaliscodesapariciones.rys2000dev.workers.dev//municipios.geojson...
    ✅ GeoJSON con 6624 featur